# Multi-model metric distributions at 300 tokens

Compare four models for perplexity, D-score, prompt mean, and prompt standard deviation. Each figure keeps the dataset rows and HG/MG columns from the reference distribution plots.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebooks.utils.logprob_metrics_distributions_utils import (
    default_length_metrics_path,
    load_length_metric_table,
)

sns.set_theme(style="whitegrid", context="notebook")

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
PLOT_OUTPUT_ROOT = DATA_ROOT / "plots" / "multi_model_metric_distribution"
PLOT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LENGTH_METRICS_PATH = default_length_metrics_path(PROCESSED_ROOT)

## Parameters

In [ ]:
TOKEN_LENGTH = 300
SELECTED_MODELS = [
    "gemma_3_1b_it",
    "gemma_3_270m_it",
    "littlelamb",
    "smollm2_360m",
]
METRICS = [
    "perplexity",
    "d_score_top20",
    "prompt_mu_score",
    "prompt_sigma_score",
]
MODEL_PALETTE = dict(zip(SELECTED_MODELS, sns.color_palette("viridis", len(SELECTED_MODELS))))
SPLIT_ORDER = ["HG", "MG"]

In [ ]:
length_metric_df = load_length_metric_table(LENGTH_METRICS_PATH)
plot_df = length_metric_df.loc[
    length_metric_df["model"].isin(SELECTED_MODELS)
    & length_metric_df["token_length"].eq(TOKEN_LENGTH)
].copy()
plot_df["model"] = pd.Categorical(plot_df["model"], categories=SELECTED_MODELS, ordered=True)
datasets = sorted(plot_df["dataset"].dropna().unique())
print(f"Loaded {len(plot_df):,} rows at token length {TOKEN_LENGTH}")
print(f"Models: {', '.join(SELECTED_MODELS)}")
print(f"Datasets: {', '.join(datasets)}")

## Distribution plots

In [ ]:
for metric in METRICS:
    metric_df = plot_df.dropna(subset=[metric]).copy()
    metric_df[metric] = pd.to_numeric(metric_df[metric], errors="coerce")
    metric_df = metric_df.dropna(subset=[metric])
    if metric_df.empty:
        print(f"Skipping {metric}: no values at token length {TOKEN_LENGTH}")
        continue

    grid = sns.FacetGrid(
        metric_df,
        row="dataset",
        col="split",
        hue="model",
        row_order=datasets,
        col_order=SPLIT_ORDER,
        hue_order=SELECTED_MODELS,
        palette=MODEL_PALETTE,
        height=1.65,
        aspect=2.4,
        margin_titles=False,
        sharex=True,
        sharey=False,
    )
    grid.map_dataframe(
        sns.kdeplot,
        x=metric,
        fill=False,
        linewidth=1.8,
        warn_singular=False,
    )
    grid.set_titles("")
    grid.figure.subplots_adjust(hspace=0.55, top=0.86)
    grid.figure.canvas.draw()
    for row_index, dataset in enumerate(datasets):
        for col_index, split in enumerate(SPLIT_ORDER):
            grid.axes[row_index, col_index].set_title(
                split if row_index == 0 else "",
                pad=8,
            )
        left_axis = grid.axes[row_index, 0].get_position()
        right_axis = grid.axes[row_index, -1].get_position()
        grid.figure.text(
            (left_axis.x0 + right_axis.x1) / 2,
            left_axis.y1 + 0.035,
            dataset,
            ha="center",
            va="bottom",
            fontsize=10,
        )
    if metric == "perplexity":
        grid.set(xscale="log")
    grid.set_axis_labels(metric, "Density")
    grid.add_legend(title="Model")
    grid.figure.subplots_adjust(top=0.86)
    grid.figure.suptitle(
        f"{metric}: distribution progression at {TOKEN_LENGTH} tokens",
        fontsize=15,
        fontweight="bold",
    )
    output_path = PLOT_OUTPUT_ROOT / f"ridgeline_{metric}__all_models__{TOKEN_LENGTH}_tokens.png"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    grid.figure.savefig(
        str(output_path),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

## Distribution summary tables

Each table reports the distribution mean and standard deviation in separate columns for every dataset, LLM, and HG/MG split combination.

In [ ]:
from IPython.display import display

for metric in METRICS:
    metric_df = plot_df.copy()
    metric_df[metric] = pd.to_numeric(metric_df[metric], errors="coerce")
    metric_df = metric_df.dropna(subset=[metric])
    summary = (
        metric_df.groupby(["dataset", "model", "split"], observed=True)[metric]
        .agg(mean="mean", std="std")
        .reindex(
            pd.MultiIndex.from_product(
                [datasets, SELECTED_MODELS, SPLIT_ORDER],
                names=["dataset", "model", "split"],
            )
        )
    )
    table = summary.unstack(["model", "split"])
    table.columns = table.columns.set_names(["Statistic", "LLM", "Split"])
    table = table.reorder_levels(["LLM", "Split", "Statistic"], axis=1)
    table = table.sort_index(axis=1, level=["LLM", "Split"])
    table.index.name = "Dataset"
    table.to_csv(PLOT_OUTPUT_ROOT / f"summary_{metric}__{TOKEN_LENGTH}_tokens.csv")
    display(table.style.format("{:.4g}").set_caption(f"{metric}: distribution mean and standard deviation at {TOKEN_LENGTH} tokens"))